In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
!pip install -q kagglehub torch torchvision pandas numpy matplotlib seaborn scikit-learn pillow

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)

In [ ]:
import kagglehub

path = kagglehub.dataset_download(
    "kmader/skin-cancer-mnist-ham10000"
)

print("Path to dataset files:", path)

In [ ]:
import os

print("Dataset path:", path)

for root, dirs, files in os.walk(path):
    print("\nFolder:", root)

    for file in files[:10]:
        print("  ", file)

    if len(files) > 10:
        print("  ...")

In [ ]:
metadata_path = os.path.join(path, "HAM10000_metadata.csv")

df = pd.read_csv(metadata_path)

print(df.head())

In [ ]:
print(df.shape)

In [ ]:
print(df.columns)

In [ ]:
image_dirs = []

for root, dirs, files in os.walk(path):
    for file in files:
        if file.lower().endswith((".jpg", ".jpeg", ".png")):
            image_dirs.append(root)
            break

print("Image directories:")
for directory in image_dirs:
    print(directory)

In [ ]:
image_paths = {}

for root, dirs, files in os.walk(path):
    for file in files:
        if file.lower().endswith((".jpg", ".jpeg", ".png")):
            image_id = os.path.splitext(file)[0]
            image_paths[image_id] = os.path.join(root, file)

print("Total images found:", len(image_paths))

In [ ]:
df["image_path"] = df["image_id"].map(image_paths)

print(df.head())

In [ ]:
print("Missing image paths:", df["image_path"].isna().sum())

In [ ]:
print(df["dx"].value_counts())

In [ ]:
plt.figure(figsize=(10, 5))

sns.countplot(
    data=df,
    x="dx",
    order=df["dx"].value_counts().index
)

plt.title("HAM10000 Class Distribution")
plt.xlabel("Class")
plt.ylabel("Number of Images")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 6))

for ax, (_, row) in zip(axes.flat, df.sample(10).iterrows()):

    image = Image.open(row["image_path"])

    ax.imshow(image)
    ax.set_title(row["dx"])
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
class_names = sorted(df["dx"].unique())

print(class_names)

In [ ]:
class_to_idx = {
    class_name: idx
    for idx, class_name in enumerate(class_names)
}

print(class_to_idx)

In [ ]:
df["label"] = df["dx"].map(class_to_idx)

print(df[["dx", "label"]].head())

In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label"],
    random_state=42
)

In [ ]:
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)

In [ ]:
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
class HAM10000Dataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):

        row = self.dataframe.iloc[idx]

        image = Image.open(row["image_path"]).convert("RGB")

        label = row["label"]

        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
train_dataset = HAM10000Dataset(
    train_df,
    transform=train_transform
)

val_dataset = HAM10000Dataset(
    val_df,
    transform=test_transform
)

test_dataset = HAM10000Dataset(
    test_df,
    transform=test_transform
)

In [ ]:
batch_size = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2
)

In [ ]:
images, labels = next(iter(train_loader))

print("Image shape:", images.shape)
print("Labels shape:", labels.shape)

In [ ]:
model = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

In [ ]:
model.fc

In [ ]:
num_classes = len(class_names)

model.fc = nn.Linear(
    model.fc.in_features,
    num_classes
)

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using:", device)

model = model.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()

In [ ]:
criterion = nn.CrossEntropyLoss()

In [ ]:
optimizer = optim.AdamW(
    model.parameters(),
    lr=0.0001,
    weight_decay=0.0001
)

In [ ]:
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.1,
    patience=2
)

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item() * images.size(0)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_accuracy = correct / total

    return epoch_loss, epoch_accuracy

In [ ]:
def validate(model, loader, criterion, device):

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_accuracy = correct / total

    return epoch_loss, epoch_accuracy

In [ ]:
num_epochs = 10

train_losses = []
val_losses = []

train_accs = []
val_accs = []

best_val_loss = float("inf")

for epoch in range(num_epochs):

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    val_loss, val_acc = validate(
        model,
        val_loader,
        criterion,
        device
    )

    scheduler.step(val_loss)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f} "
        f"Train Acc: {train_acc:.4f} "
        f"Val Loss: {val_loss:.4f} "
        f"Val Acc: {val_acc:.4f}"
    )

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save(
            model.state_dict(),
            "best_resnet18_ham10000.pth"
        )

        print("Best model saved!")

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")

plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(train_accs, label="Train Accuracy")
plt.plot(val_accs, label="Validation Accuracy")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy")

plt.legend()
plt.show()

In [ ]:
model.load_state_dict(
    torch.load(
        "best_resnet18_ham10000.pth",
        map_location=device
    )
)

model = model.to(device)

In [ ]:
model.eval()

all_labels = []
all_predictions = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)

        outputs = model(images)

        _, predictions = torch.max(outputs, 1)

        all_labels.extend(labels.numpy())
        all_predictions.extend(
            predictions.cpu().numpy()
        )

In [ ]:
accuracy = accuracy_score(
    all_labels,
    all_predictions
)

print("Test Accuracy:", accuracy)

In [ ]:
print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=class_names
    )
)

In [ ]:
cm = confusion_matrix(
    all_labels,
    all_predictions
)

plt.figure(figsize=(8, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("HAM10000 Confusion Matrix")

plt.show()

In [ ]:
def predict_image(image_path, model, transform, class_names, device):

    model.eval()

    image = Image.open(image_path).convert("RGB")

    image_tensor = transform(image).unsqueeze(0)

    image_tensor = image_tensor.to(device)

    with torch.no_grad():

        output = model(image_tensor)

        probabilities = torch.softmax(output, dim=1)

        confidence, predicted = torch.max(
            probabilities,
            1
        )

    predicted_class = class_names[predicted.item()]

    return predicted_class, confidence.item()

In [ ]:
# Predict on an image already present in the HAM10000 dataset.
# This avoids FileNotFoundError from the placeholder "your_image.jpg".

image_path = df["image_path"].dropna().iloc[0]

predicted_class, confidence = predict_image(
    image_path,
    model,
    test_transform,
    class_names,
    device
)

print("Image path:", image_path)
print("Predicted class:", predicted_class)
print(f"Confidence: {confidence * 100:.2f}%")

# Display the image with its predicted class
image = Image.open(image_path).convert("RGB")
plt.figure(figsize=(5, 5))
plt.imshow(image)
plt.title(f"Prediction: {predicted_class}")
plt.axis("off")
plt.show()

# To predict on your own image in Google Colab instead, run this in a new cell:
# from google.colab import files
# uploaded = files.upload()
# image_path = next(iter(uploaded))
# predicted_class, confidence = predict_image(
#     image_path, model, test_transform, class_names, device
# )
# print("Predicted class:", predicted_class)
# print(f"Confidence: {confidence * 100:.2f}%")
